# 03: 量子センサのフィードバック安定化
### (Feedback Stabilization of Quantum Sensors)

---

## 1. Thesis: 科学的問いと仮説

量子センサ、特にNVセンターや原子磁気センサなどは非常に高い感度を持ちますが、同時に温度変化や外部磁場の揺らぎ（ドリフト）などの環境ノイズの影響を強く受けます。これらのノイズはセンサの「動作点」をずらし、測定感度を低下させたり、測定値にバイアスを与えたりします。

**問い** : 環境ドリフトが刻一刻と変化する状況において、測定結果をリアルタイムで監視し、それに基づいてセンサのパラメータを修正する「フィードバック制御」を導入することで、センサの感度を常に最大の状態（最適動作点）に維持し続けることは可能か？

**仮説** : 測定確率が最も急峻に変化する「クアドラチャ（$P=0.5$）」を動作点とし、そこからのズレを打ち消すように補正位相をフィードバック適用することで、大きなドリフトが存在しても実質的なセンシング精度を安定化させることができる。


## 2. Theoretical Background (理論的背景)

### 2.1 最適動作点（クアドラチャ）
量子センシングの基本的な出力は $P(\theta) = \cos^2(\theta/2)$ です。位相 $\theta$ に対する感度はその微分 $dP/d\theta$ で決まります。
- **$\theta \approx 0$**: $dP/d\theta \approx 0$。位相が少し変わっても確率はほとんど変わらず、最も感度が低い状態。
- **$\theta \approx \pi/2$**: $dP/d\theta$ が最大。確率 $P=0.5$ の付近は「クアドラチャ」と呼ばれ、最も微小な位相変化を検出しやすいポイントです。

### 2.2 ドリフトとフィードバックの原理
現実の環境では、時間 $t$ とともに未知のドリフト $\theta_{drift}(t)$ が加わります。
これを放置すると、センサの動作点はクアドラチャから外れていきます。

フィードバック制御では、以下のループを回します：
1. **測定**: 現在の状態をサンプリングし、確率（または移動平均）を計算。
2. **誤差評価**: 目標値 $P=0.5$ との差分 $e$ を算出。
3. **補正**: 補正位相 $\theta_{fb}(t)$ を更新。
   $$\theta_{fb}(t+1) = \theta_{fb}(t) - G \cdot (P_{measured} - 0.5)$$
   （ここで $G$ はフィードバックゲイン）

この合成位相 $\theta_{total} = \theta_{signal} + \theta_{drift} + \theta_{fb}$ を常にクアドラチャ付近に拘束することが本アルゴリズムの目的です。

## 3. 実装 (Implementation)

環境ドリフトが存在する中で、適応的に補正位相を更新していくシミュレーションを実装します。時間の経過とともにドリフトが増大していくケースで、フィードバックの有無が測定結果にどう影響するかを確認します。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

def simulate_sensing_with_feedback(steps: int = 100, gain: float = 1.0, drift_strength: float = 0.05):
    """環境ドリフト下での量子センシングとフィードバック安定化をシミュレートします。"""
    
    sim = AerSimulator()
    
    # ドリフトの履歴（ランダムウォーク的なドリフトを生成）
    true_drift = np.cumsum(np.random.normal(0, drift_strength, steps))
    
    # 記録用
    measured_probs_no_fb = []
    measured_probs_with_fb = []
    applied_fb_phase = np.zeros(steps)
    total_phase_with_fb = []

    # 1. フィードバックなしのケース
    for t in range(steps):
        phase = np.pi/2 + true_drift[t] # クアドラチャからスタートしてドリフト
        qc = QuantumCircuit(1, 1)
        qc.h(0)
        qc.rz(phase, 0)
        qc.h(0)
        qc.measure(0, 0)
        
        # 100ショット程度で現在の確率を評価（現実のセンサに近い挙動）
        res = sim.run(transpile(qc, sim), shots=100).result().get_counts()
        prob = res.get('1', 0) / 100
        measured_probs_no_fb.append(prob)

    # 2. フィードバックありのケース
    current_fb = 0.0
    for t in range(steps):
        # 合成位相 = クアドラチャ + ドリフト + 現在までのフィードバック補正
        phase = np.pi/2 + true_drift[t] + current_fb
        
        qc = QuantumCircuit(1, 1)
        qc.h(0)
        qc.rz(phase, 0)
        qc.h(0)
        qc.measure(0, 0)
        
        res = sim.run(transpile(qc, sim), shots=100).result().get_counts()
        prob = res.get('1', 0) / 100
        measured_probs_with_fb.append(prob)
        total_phase_with_fb.append(phase)
        
        # フィードバック更新: P=0.5 (100ショット中50回) からのズレを打ち消す
        error = prob - 0.5
        current_fb -= gain * error
        applied_fb_phase[t] = current_fb

    return true_drift, measured_probs_no_fb, measured_probs_with_fb, applied_fb_phase

# シミュレーション実行
steps = 80
drift_data, prob_no_fb, prob_with_fb, fb_phase = simulate_sensing_with_feedback(steps=steps, gain=1.2)



## 4. Visualization & Analysis (可視化と解析)

左側のグラフは環境ドリフトそのものと、それに対してフィードバックがどう「逆方向」に追従したかを示します。右側のグラフは、実際の測定確率の変化です。


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# --- グラフ1: ドリフト vs フィードバック補正 ---
ax1.plot(drift_data, label='True Environmental Drift', color='#d62728', lw=2)
ax1.plot(-fb_phase, '--', label='Feedback Compensation', color='#1f77b4', lw=2)
ax1.set_title("Environmental Drift and Feedback Tracking")
ax1.set_xlabel("Time Step")
ax1.set_ylabel("Phase Offset [rad]")
ax1.legend()
ax1.grid(True, alpha=0.3)

# --- グラフ2: 測定確率の安定性 ---
ax2.axhline(0.5, ls=':', color='gray', label='Ideal Quadrature (P=0.5)')
ax2.plot(prob_no_fb, 'o-', ms=4, label='Without Feedback', color='#ff7f0e', alpha=0.6)
ax2.plot(prob_with_fb, 'o-', ms=4, label='With Feedback', color='#2ca02c')
ax2.set_title("Measurement Stability (Probability)")
ax2.set_xlabel("Time Step")
ax2.set_ylabel("Probability P(1)")
ax2.set_ylim(0, 1)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 5. Conclusion & Future Work (結論と展望)

### 結論
実験結果から、以下の重要なポイントが確認されました：

1. **ドリフトの打ち消し効果** :  
フィードバック制御を導入することで、ランダムに変化する環境ドリフトに対して補正位相が追従し、合成位相をクアドラチャ（高感度ポイント）に引き戻し続けることができました。

2. **安定性の維持** :  
フィードバックなしの状態では測定確率が 0 や 1 に張り付いてしまい感度を失っていましたが、フィードバックありでは常に $P=0.5$ 付近を維持し、微小な変化を検出し続けられる状態をキープできました。

### 展望

- **最適制御理論（PID / Kalman Filter）** :  
今回は単純な比例ゲインのみを使用しましたが、ノイズの特性に合わせて積分(I)や微分(D)を加えたPID制御、あるいはカルマンフィルタを用いてドリフトを予測的に推定することで、より高精度な安定化が可能になります。

- **動的回路（Dynamic Circuits）への実装** :  
実機の量子コンピュータ（IBMQ等）では、測定結果をその回路内で即座に条件分岐（if文）に使える機能があります。これを利用して、計算の途中でエラーを自己修復する自律型量子センサの構築が次のステップとなります。
